# Solutions

## Problem 2: Feature Engineering with the Penguins Dataset

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load the dataset
df = sns.load_dataset('penguins')

### Part 1: Imputation (20pts)

In [3]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load the dataset
df = sns.load_dataset('penguins')

# (5 pts)
print(df.isnull().sum())

# (5 pts)
mode_sex = df['sex'].mode()[0]
df['sex'] = df['sex'].fillna(mode_sex)

# (5 pts)
num_cols = ['bill_length_mm', 'bill_depth_mm',
            'flipper_length_mm', 'body_mass_g']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# (5 pts)
print(df.isnull().sum())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


### Part 2: Selection (15 pts)

In [4]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load the dataset
df = sns.load_dataset('penguins')

# (5 pts)
df_num = df.select_dtypes(include=np.number)

# (5 pts)
corr_with_target = (df_num.corr()['body_mass_g']
                    .abs()
                    .sort_values(ascending=False))
print(corr_with_target)

# (5 pts)
# flipper_length_mm is MOST correlated with body_mass_g (~0.87).
# bill_depth_mm is LEAST correlated with body_mass_g (~0.14),
# excluding body_mass_g itself.

body_mass_g          1.000000
flipper_length_mm    0.871202
bill_length_mm       0.595110
bill_depth_mm        0.471916
Name: body_mass_g, dtype: float64


### Part 3: Transformation (25pts)

In [5]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load the dataset
df = sns.load_dataset('penguins')

from sklearn.preprocessing import StandardScaler

# (5 pts)
df['body_mass_log'] = np.log1p(df['body_mass_g'])
print(df[['body_mass_g', 'body_mass_log']].head())

# (5 pts)
scaler = StandardScaler()
df['flipper_std'] = scaler.fit_transform(df[['flipper_length_mm']])
print(f"flipper_std mean: {df['flipper_std'].mean():.4f}")
print(f"flipper_std std:  {df['flipper_std'].std():.4f}")

# (5 pts)
df['mass_category'] = pd.cut(
    df['body_mass_g'],
    bins=[0, 3500, 5000, float('inf')],
    labels=['light', 'medium', 'heavy']
)
print(df['mass_category'].value_counts())

# (5 pts)
# Log transforms compress large values and spread out small
# ones, pulling a skewed distribution closer to normal.
# Linear regression works better when features are not
# dominated by extreme values.

# (5 pts)
# Standardization rescales a feature to mean=0 and std=1.
# Many ML algorithms (KNN, SVM, PCA) are sensitive to scale,
# so without it, larger-valued features unfairly dominate.

   body_mass_g  body_mass_log
0       3750.0       8.229778
1       3800.0       8.243019
2       3250.0       8.086718
3          NaN            NaN
4       3450.0       8.146419
flipper_std mean: -0.0000
flipper_std std:  1.0015
mass_category
medium    203
light      78
heavy      61
Name: count, dtype: int64


### Part 4: Creation (25pts)

In [6]:
import seaborn as sns
import pandas as pd
import numpy as np

# Load the dataset
df = sns.load_dataset('penguins')

from sklearn.preprocessing import LabelEncoder

# (5 pts)
df['bill_ratio'] = df['bill_length_mm'] / df['bill_depth_mm']
print(df[['bill_length_mm', 'bill_depth_mm', 'bill_ratio']].head())

# (5 pts)
df = pd.get_dummies(df, columns=['island'], dtype=int)
island_cols = [c for c in df.columns if 'island' in c.lower()]
print(df[island_cols].head())

# (5 pts)
df['is_heavy'] = (df['body_mass_g'] >= 5000).astype(int)
print(df['is_heavy'].value_counts())

# (5 pts)
le = LabelEncoder()
df['species_label'] = le.fit_transform(df['species'])
print(df[['species', 'species_label']].drop_duplicates().sort_values('species_label'))

# (5 pts)
# One-hot encoding creates a separate binary column for each
# category — use for categories with NO natural order (like
# island names). Label encoding assigns a single integer —
# use for categories WITH a natural order (like S < M < L).

   bill_length_mm  bill_depth_mm  bill_ratio
0            39.1           18.7    2.090909
1            39.5           17.4    2.270115
2            40.3           18.0    2.238889
3             NaN            NaN         NaN
4            36.7           19.3    1.901554
   island_Biscoe  island_Dream  island_Torgersen
0              0             0                 1
1              0             0                 1
2              0             0                 1
3              0             0                 1
4              0             0                 1
is_heavy
0    277
1     67
Name: count, dtype: int64
       species  species_label
0       Adelie              0
152  Chinstrap              1
220     Gentoo              2


### Part 5: Extraction (15pts)

In [10]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Load the dataset
df = sns.load_dataset('penguins')

# (5 pts)
features = ['bill_length_mm', 'bill_depth_mm',
            'flipper_length_mm', 'body_mass_g']
df_pca_input = df[features].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca_input)

# (5 pts)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
print(df_pca.head())

# (5 pts)
print(f"Variance per component: {pca.explained_variance_ratio_.round(4)}")
print(f"Total variance kept:    {pca.explained_variance_ratio_.sum():.4f}")

# The 2 components capture about 88% of the total variance
# from the original 4 features — most of the information is
# kept while cutting features in half.

        PC1       PC2
0 -1.843445  0.047702
1 -1.306762 -0.428348
2 -1.369181 -0.154476
3 -1.878827 -0.002048
4 -1.911748  0.829210
Variance per component: [0.6884 0.1931]
Total variance kept:    0.8816
